#### __libraries__

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [7]:
from shared_scripts.statistical_analysis import t_margin_of_error

In [10]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

#### __plt params__

In [15]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [68]:
def _2C_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
        
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in data.dtypes):
        raise TypeError("All columns in inout must be numerical")
        
    #assign timepoint
    frq= kwargs.get('frequency', np.NaN)
    indel= kwargs.get('initial_delay', np.NaN)
    if "frequency" not in kwargs or "initial_delay" not in kwargs:
        raise ValueError("frequency and initial_delay must be provided")
    try:
        data= data.assign(timepoint_minutes=data['frame'] * frq + indel).drop(columns=["frame"])
    except Exception as ex:
        raise ValueError("failed to calculate the timepoints in minutes")
        
    #set the index 
    if 'timepoint_minutes' in data.columns:
        data= data.set_index('timepoint_minutes')
    else:
        raise ValueError("dataset is missing a frame column")
        
    #calculate the mean
    try:
        data= data.assign(avg_distance_ctrl= data.iloc[:, 0:3].mean(axis= 1),
                          avg_distance_lata= data.iloc[:, 3:6].mean(axis= 1))
    except Exception as ex:
        print(f'failed to calculate the mean: {ex}')
        data= data.assign(avg_distance_ctrl= np.NaN,
                          avg_distance_lata= np.NaN)
        
    #calculate the MOE95
    try:
        data= data.assign(moe95_ctrl= data.apply(lambda x: t_margin_of_error(x.iloc[0:3]), axis= 1),
                          moe95_lata= data.apply(lambda x: t_margin_of_error(x.iloc[3:6]), axis= 1))
    except Exception as ex:
        print(f'failed to calculate the moe95: {ex}')
        data= data.assign(moe95_ctrl= np.NaN,
                          moe95_lata= np.NaN)
        
    return data

#### __data load__

In [24]:
_6C= BB2_load_into_dataframe('Figure_6C.csv')

#### __data processing__

In [70]:
_6C_final= _2C_data_process(_6C, frequency= 6, initial_delay= 12)

In [74]:
_6C_final.head()

,avg_dist_per_frame_D14_dmso_ctrl,avg_dist_per_frame_E14_dmso_ctrl,avg_dist_per_frame_F14_dmso_ctrl,avg_dist_per_frame_D17_lata,avg_dist_per_frame_F17_lata,avg_dist_per_frame_E17_lata,avg_distance_ctrl,avg_distance_lata,moe95_ctrl,moe95_lata
timepoint_minutes,,,,,,,,,,
18,5.765752,5.615958,4.852404,5.365338,4.677185,5.194730,5.411371,5.079085,1.216828,0.890206
24,5.501720,5.897168,6.258467,5.196473,5.305220,4.850155,5.885785,5.117283,0.940250,0.590339
30,6.034429,5.274513,5.502149,4.824812,4.644690,4.441496,5.603697,4.636999,0.968821,0.476392
36,5.315914,4.803438,4.941164,4.117481,3.913838,3.585564,5.020172,3.872294,0.658833,0.666695
42,5.269997,4.283253,4.294356,3.204702,3.300422,2.922690,4.615869,3.142605,1.407312,0.487817
